# Results
Loads the trained model and plots its predictions **on the validation set**.

Requires the installed package (`pip install -e .`) and a trained model in `models/` (`python -m spotforecast.train`).

In [ ]:
#%load_ext autoreload
#%autoreload 2
# the autoreload magic is convenient when using a notebook while modifying the code in imported modules
# it also slows things down (because it re-imports the modules all the time), so you can just comment this out

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()
# Note: check with !nvidia-smi if there really are no available GPUs, can also be a PyTorch problem!
# try changing the line `- pytorch` to `- pytorch-gpu` in environment.yml when using an older GPU and/or older driver 

In [ ]:
import torch
import secrets
from spotforecast.utils import seed_torch

# Draw random seeds using Python's cryptographic library and cut them down to size
_global_seed = secrets.randbits(128) % (2**64-1)
_local_seed = secrets.randbits(128) % (2**64-1)

# If you want to reproduce a previous experiment (and only then) set the seeds to that previously recorded value
# and set `reproducible=True`. N.B.: using `42` is never a good idea!
seed_torch(_global_seed, reproducible=False)
_rng_cpu = torch.Generator()
_rng_cpu = _rng_cpu.manual_seed(_local_seed)

In [ ]:
from spotforecast.config import PROJECT_ROOT, load_config
from spotforecast.data.dataset import load_datasets, get_feature_dims, split_history_future_batch_transform
from spotforecast.models.lstm import build_model
from spotforecast.utils import load_model
from torch.utils.data import DataLoader
import torch.nn as nn

config = load_config("lstm")
data_config = load_config("data")

train_ds, valid_ds = load_datasets(config, data_config)

train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, pin_memory=torch.cuda.is_available(), generator=_rng_cpu)
val_loader = DataLoader(valid_ds, batch_size=config['batch_size'], shuffle=False, pin_memory=torch.cuda.is_available(), generator=_rng_cpu)

n_hist_features, n_future_features = get_feature_dims(train_ds)
model = build_model(config, n_hist_features, n_future_features, rng=_rng_cpu).to(device)
model.eval()
criterion = nn.MSELoss(reduction="mean")
batch_tsfm_fn = split_history_future_batch_transform
save_model_path = PROJECT_ROOT / "models" / f"{config['model_name']}.pth"

In [ ]:
print(f"Loading trained model...\n")
load_model(model_path=save_model_path, model=model, optimizer=None, device=device, with_opt=False)
print(f"Model architecture:\n{model}")

In [ ]:
from spotforecast.plotting import predict_one_sample, plot_one_prediction

x, y = valid_ds[425]
pred, y = predict_one_sample(model, x, y, device, transform_batch_fn=batch_tsfm_fn)
fig = plot_one_prediction(pred, y)
fig.show()

In [ ]:
x, y = valid_ds[44]
pred, y = predict_one_sample(model, x, y, device, transform_batch_fn=batch_tsfm_fn)
fig = plot_one_prediction(pred, y)
fig.show()

In [ ]:
x, y = valid_ds[43]
pred, y = predict_one_sample(model, x, y, device, transform_batch_fn=batch_tsfm_fn)
fig = plot_one_prediction(pred, y)
fig.show()

In [ ]:
x, y = valid_ds[42]
pred, y = predict_one_sample(model, x, y, device, transform_batch_fn=batch_tsfm_fn)
fig = plot_one_prediction(pred, y)
fig.show()

In [ ]:
x, y = valid_ds[41]
pred, y = predict_one_sample(model, x, y, device, transform_batch_fn=batch_tsfm_fn)
fig = plot_one_prediction(pred, y)
fig.show()